# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates loading, overviewing, processing, and visualizing a dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The dataset includes survey data on mental health among residents of Kilifi County, Kenya.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their entity `@id`s. Use `mlcroissant` to enumerate the record sets and their fields, referencing entities by their `@id`.

In [ ]:
# List record sets and their fields using `@id`
record_sets = list(dataset.record_sets)
print("Available record sets:")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']} | Name: {rs['name']}")
    if 'field' in rs:
        print("  Fields:")
        for f in rs['field']:
            print(f"    Field @id: {f['@id']} | Name: {f['name']}")
    if 'column' in rs:
        print("  Columns:")
        for c in rs['column']:
            print(f"    Column @id: {c['@id']} | Name: {c['name']}")
print("\nExample record from first record set:")
if record_sets:
    example_rs_id = record_sets[0]['@id']
    for x in dataset.records(record_set=example_rs_id):
        pprint(x)
        break

## 3. Data Extraction
Extract and load the records from specified record sets as DataFrames for analysis. Use the record set and field `@id`s from above.

In [ ]:
# Collect all record sets @id for extraction
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nDataFrame for RecordSet @id: {record_set_id}")
    print(df.columns.tolist())
    print(df.head())
# For subsequent EDA, choose the main survey record set
main_record_set_id = record_set_ids[0]
main_df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing with clear reference to field `@id`s.

- Filter records by threshold (e.g., GAD-7 or PHQ-9 scores).
- Normalize numeric fields (e.g., PHQ-9 score).
- Group by demographics (e.g., gender or education level).

Each entity (record set, field, column) is referenced by its `@id`.

In [ ]:
# Example: Filter by PHQ-9 score ('phq9_score' column), referenced by its @id
# Find the column @id for PHQ-9 score
phq9_col_id = None
gender_col_id = None
for rs in record_sets:
    if rs['@id'] == main_record_set_id:
        if 'column' in rs:
            for col in rs['column']:
                if col['name'].lower() == 'phq9_score':
                    phq9_col_id = col['@id']
                if col['name'].lower() == 'gender':
                    gender_col_id = col['@id']
            break
print(f"PHQ-9 column @id: {phq9_col_id}")
print(f"Gender column @id: {gender_col_id}")

# If DataFrame columns are mapped to column @id, ensure correct extraction
if phq9_col_id is None:
    phq9_col_id = 'phq9_score'  # fallback to string name
if gender_col_id is None:
    gender_col_id = 'gender'

# Filter for high PHQ-9 scores, e.g., >10
threshold = 10
filtered_df = main_df[main_df[phq9_col_id] > threshold]
print(f"Filtered records with {phq9_col_id} > {threshold}:")
print(filtered_df.head())

# Normalize PHQ-9 score
filtered_df[f"{phq9_col_id}_normalized"] = (filtered_df[phq9_col_id] - filtered_df[phq9_col_id].mean()) / filtered_df[phq9_col_id].std()
print(f"Normalized {phq9_col_id} for filtered records:")
print(filtered_df[[phq9_col_id, f"{phq9_col_id}_normalized"]].head())

# Group by gender (@id reference)
if gender_col_id in main_df.columns:
    grouped_df = filtered_df.groupby(gender_col_id)[phq9_col_id].mean().reset_index()
    print(f"Grouped average {phq9_col_id} by {gender_col_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields using their `@id`.

- Plot normalized PHQ-9 scores.
- Compare PHQ-9 by gender.

In [ ]:
# Visualization imports
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of normalized PHQ-9 scores
plt.figure(figsize=(8,5))
sns.histplot(filtered_df[f"{phq9_col_id}_normalized"], bins=20, kde=True)
plt.title(f"Distribution of normalized PHQ-9 scores ({phq9_col_id})")
plt.xlabel(f"Normalized PHQ-9 score (@id: {phq9_col_id})")
plt.ylabel("Count")
plt.show()

# Compare PHQ-9 scores by gender
plt.figure(figsize=(8,5))
sns.boxplot(x=gender_col_id, y=phq9_col_id, data=filtered_df)
plt.title(f"PHQ-9 Scores by Gender (@id: {gender_col_id})")
plt.xlabel(f"Gender (@id: {gender_col_id})")
plt.ylabel(f"PHQ-9 score (@id: {phq9_col_id})")
plt.show()

## 6. Conclusion
We explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, loading it using `mlcroissant` and referencing all entities by their `@id`. After filtering for high PHQ-9 scores and normalizing, we visualized distributions and differences across gender.

**Key findings:**
- The dataset covers a rich set of mental health indicators and demographics using survey data.
- Analysis by unique Croissant entity `@id` ensures reproducible, standards-based data science workflows.
- The visualizations highlighted potential gender variation and score distributions among participants.

Further work could include correlation analysis between other fields (e.g. GAD-7, PSQ scores), deeper demographic breakdown, and community health strategy development using the AI-ready structure.